<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Timbre/Style transfer</b> </br> 
We will select a sound (encoded as a .ecdc encodec file) and "play it through" a trained RNeNcodec model.  
That is, we drive the sound with an imput audio + parametes (whatever the trained model exposes). Instead of "autoregressive" generation, we take the output from the model and render it to auio, but don't feed it back as input to the next step - instead we take the next input step from the input audio. The model is "forced", but maintains state in the same way that RNNs always do.   
  
You will have a GUI file selector (point it to the root of the system you want to select coded sounds from).  
You will also provide a trained model to load from a checkpoint.  

In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

In [ ]:
import librosa
import torch
import os
import numpy as np

import matplotlib.pyplot as plt
from IPython.display import Audio, display
import random

import torchaudio
#import seaborn as sns
#from transformers import EncodecModel, AutoProcessor
import IPython.display as ipd
from pathlib import Path
import pickle

from transformers import EncodecModel

import tkinter as tk
from tkinter import filedialog

from rnencodec.audioDataLoader.audio_dataset import efficient_codes_to_latents

sample_rate=24000

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>Parameter selection</b> </br>  
file_path  @ the root of the folder to use for the FUI file sector </br>
file_name  @ either a file in that path, or None if you want to select with the GUI   </br>
</br>
run_directory @ the name of the folder that contains your checkpoints directory  </br>
checkpoint_fname @ the name of the saved .pt checkpoint for the trained model  </br>

In [ ]:
# Cell 1: Set file path and parameters
# Configure your dataset path and parameters here

# This should be a path to a collection of .ecdc files you want to explore
file_path = "/slowdisk/data/"  # Update this path
# If file_name == None, you will be able to choose your file in a graphical "file picker"
file_name = "DSPistons--rate_exp-00.75--c-02--x-92.ecdc"  # Update this filename
file_name = None

# and the model you want to play sounds through
run_directory = str(Path('output/20260205_183830_datapreptest_dynamic_forkad'))
checkpoint_fname =   "last_checkpoint.pt" #"checkpoint_125.pt" # "checkpoint_25.pt" # "checkpoint_25.pt" #


In [ ]:
# Don't mess with these unlesss you know what you are doing:
config_path = os.path.join(run_directory, "config_v2.pt")
checkpoint_path = os.path.join(run_directory, "checkpoints", checkpoint_fname ) # "last_checkpoint.pt") # "checkpoint_40.pt") # 

<div style="width: 100%; height: 20px; background-color: blue;"></div>
<b>Helper functions</b> </br>   

In [ ]:
def pick_file():
    root = tk.Tk()
    root.withdraw()
    return filedialog.askopenfilename(
        title="Select .ecdc file",
        initialdir=file_path,
        filetypes=[("ECDC files", "*.ecdc"), ("All files", "*.*")]
    )

In [ ]:
def load_and_examine_tokens(ecdcpath):
    """Load dataset and examine a specific sample
    
    Returns
    sample, audio_codes, audio_scales, padding_mask
    sample - the data structure containing the record for indexed sample
    audio_codes - [1,8,T] assuming 8 tokens per frame were stored
    audio_scales - probably None since the 24kHz model doesn't normalize by chunk (doesn't chunk at all)
    padding_mask - also not relevant for the 24kHz model
    """

    saved_data = torch.load(ecdcpath)
    
    # Extract the data in the new simplified format
    audio_codes = saved_data['audio_codes']
    audio_scales = saved_data['audio_scales'] 
    audio_length = saved_data['audio_length']
    
    # Create padding mask from the stored audio length
    padding_mask = torch.zeros(1, audio_length, dtype=torch.bool)
    
    print("  ✅ Loaded encoder outputs")
    print(f"  Audio codes shape: {audio_codes.shape}")
    print(f"  Audio scales type: {type(audio_scales)}")
    if isinstance(audio_scales, list):
        print(f"  Audio scales list length: {len(audio_scales)}")
        if len(audio_scales) > 0 and hasattr(audio_scales[0], 'shape'):
            print(f"  Audio scales[0] shape: {audio_scales[0].shape}")
    else:
        print(f"  Audio scales shape: {audio_scales.shape}")
    print(f"  Audio length: {audio_length}")
    print(f"  Padding mask shape: {padding_mask.shape}")
    
    # Handle different possible shapes for display
    if audio_codes.dim() == 4:
        # Shape [1, 1, n_q, time] -> [1, n_q, time] 
        audio_codes = audio_codes.squeeze(1)
    elif audio_codes.dim() == 2:
        # Shape [n_q, time] -> add batch dimension
        audio_codes = audio_codes.unsqueeze(0)
    
    print(f"  Token shape for analysis: {audio_codes.shape}")  # Should be [1, 8, 375] for 6kbps
    print(f"  Number of codebooks: {audio_codes.shape[1]}")
    print(f"  Sequence length: {audio_codes.shape[2]}")
    
    return audio_codes, audio_scales, padding_mask

In [ ]:
def reconstruct_audio_progressive(audio_codes, audio_scales, padding_mask, model, codebook_counts=[2, 4, 8]):
    """Reconstruct audio using different numbers of codebooks"""
    
    reconstructions = {}
    
    for n_codebooks in codebook_counts:
        print(f"\nReconstructing with {n_codebooks} codebooks...")
        
        # Take first n codebooks and ensure batch dimension
        partial_codes = audio_codes[:, :n_codebooks, :].unsqueeze(0)  # Add batch dim
        
        # Ensure all tensors are on the same device as the model
        device = next(model.parameters()).device
        partial_codes = partial_codes.to(device)
        
        if isinstance(audio_scales, list):
            audio_scales_device = [scale.to(device) if hasattr(scale, 'to') else scale for scale in audio_scales]
        else:
            audio_scales_device = audio_scales.to(device) if hasattr(audio_scales, 'to') else audio_scales
            
        padding_mask_device = padding_mask.to(device)
        
        print(f"  Partial codes shape: {partial_codes.shape}")
        
        with torch.no_grad():
            audio_values = model.decode(partial_codes, audio_scales_device, padding_mask_device)[0]
            audio_array = audio_values.squeeze().cpu().numpy()
            
            print(f"  Output shape: {audio_array.shape}")
            print(f"  Audio length: {len(audio_array) / sample_rate:.2f} seconds")
            
            reconstructions[n_codebooks] = audio_array
    
    return reconstructions

# ------------------------------------------------------------------------------------
# one plot per audio in reconstructions

def plot_audio_comparisons(reconstructions):
    """Plot waveforms for different codebook counts"""
    
    n_plots = len(reconstructions)
    fig, axes = plt.subplots(n_plots, 1, figsize=(12, 3*n_plots))
    
    if n_plots == 1:
        axes = [axes]
    
    colors = ['blue', 'red', 'green', 'purple', 'orange']
    
    for i, (n_codebooks, audio) in enumerate(reconstructions.items()):        
        time_axis = np.linspace(0, len(audio) / sample_rate, len(audio))
        
        axes[i].plot(time_axis, audio, color=colors[i % len(colors)], 
                    linewidth=0.5, alpha=0.8)
        axes[i].set_title(f'{n_codebooks} Codebooks')
        axes[i].set_xlabel('Time (seconds)')
        axes[i].set_ylabel('Amplitude')
        axes[i].grid(True, alpha=0.3)
        
        # Add some stats
        rms = np.sqrt(np.mean(audio**2))
        axes[i].text(0.02, 0.95, f'RMS: {rms:.4f}', transform=axes[i].transAxes, 
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------------------------------
# one spectrogram per audio in reconstructions

def plot_spectrograms(reconstructions, sample_rate=24000):
    """Plot spectrograms for different codebook counts"""
    
    n_plots = len(reconstructions)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    
    if n_plots == 1:
        axes = [axes]
    
    for i, (n_codebooks, audio) in enumerate(reconstructions.items()):
        # Compute spectrogram
        D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
        
        # Plot
        img = librosa.display.specshow(D, sr=sample_rate, x_axis='time', y_axis='hz', 
                                     ax=axes[i], cmap='viridis')
        axes[i].set_title(f'{n_codebooks} Codebooks')
        
        # Add colorbar
        plt.colorbar(img, ax=axes[i], format='%+2.0f dB')
    
    fig.suptitle(f'Spectrograms', fontsize=14)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------------------------------
# One Audio Control for each audio in reconstructions
                  
def create_audio_controls(reconstructions, sample_rate=24000):
    """Create IPython Audio controls for listening"""
    
    print("\n" + "="*60)
    print("🎵 AUDIO PLAYBACK CONTROLS")
    print("="*60)
    
    for n_codebooks, audio in reconstructions.items():
        print(f"\n🎧 {n_codebooks} Codebooks :")
        print(f"   Audio length: {len(audio) / sample_rate:.2f} seconds")
        print(f"   RMS level: {np.sqrt(np.mean(audio**2)):.4f}")
        display(Audio(audio, rate=sample_rate))

In [ ]:
# Get the sequence of latents for a sequence of stacked tokens (audio_codes)
# Visualize them as heat maps

def analyze_128d_latents(audio_codes, audio_scales, padding_mask, model, n_codebooks=8, sample=None):
    """
    Reconstruct 128-D latent vectors from tokens and create waterfall plot
    
    Args:
        audio_codes: Token tensor [1, 8, time]
        model: EnCodec model
        n_codebooks: Number of codebooks to use
        sample: Sample metadata for title
    """
    
    print(f"\nAnalyzing 128-D latents with {n_codebooks} codebooks...")
    
    # Take first n codebooks
    partial_codes = audio_codes[:, :n_codebooks, :]  # [1, n, time]
    
    # Ensure tensor is on the same device as the model
    device = next(model.parameters()).device
    partial_codes = partial_codes.to(device)

    with torch.no_grad():
        # HF different from CodeCraft - no decode_talent method!!!
        # latent_embeddings = model.decode_latent(codes) # What we'd really like
        
        latent_embeddings = efficient_codes_to_latents(model, partial_codes)
        print(f' ==========   SHAPE of latent_embeddings is {latent_embeddings.shape}')
        
        # Remove batch dimension and convert to numpy
        latents = latent_embeddings.squeeze(0).cpu().numpy()  # [128, time]
        
    print(f"  Latent embeddings shape: {latents.shape}")
    print(f"  Time frames: {latents.shape[1]}")
    print(f"  Latent dimensions: {latents.shape[0]}")
    
    # Create waterfall plot
    plt.figure(figsize=(15, 8))
    
    # Plot as heatmap/waterfall
    im = plt.imshow(latents, aspect='auto', cmap='RdBu_r', interpolation='nearest')
    
    # Customize plot
    title = f'128-D Latent Vectors Over Time ({n_codebooks} codebooks)'
    if sample:
        title += f' - {sample["class_name"]}'
    plt.title(title, fontsize=14)
    
    plt.xlabel('Time Frame')
    plt.ylabel('Latent Dimension')
    plt.colorbar(im, label='Magnitude')
    
    # Add some statistics
    mean_magnitude = np.mean(np.abs(latents))
    max_magnitude = np.max(np.abs(latents))
    plt.text(0.02, 0.98, f'Mean |magnitude|: {mean_magnitude:.3f}\nMax |magnitude|: {max_magnitude:.3f}', 
             transform=plt.gca().transAxes, verticalalignment='top',
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Plot magnitude over time (RMS across latent dimensions)
    plt.figure(figsize=(12, 4))
    rms_over_time = np.sqrt(np.mean(latents**2, axis=0))  # RMS across 128 dimensions for each time frame
    time_axis = np.arange(len(rms_over_time))
    
    plt.plot(time_axis, rms_over_time, 'b-', linewidth=1.5, alpha=0.8)
    plt.title(f'RMS Magnitude of 128-D Latents Over Time ({n_codebooks} codebooks)')
    plt.xlabel('Time Frame')
    plt.ylabel('RMS Magnitude')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Show some statistics
    print(f"\n📊 Latent Statistics:")
    print(f"  Mean magnitude: {mean_magnitude:.4f}")
    print(f"  Max magnitude: {max_magnitude:.4f}")
    print(f"  Min magnitude: {np.min(np.abs(latents)):.4f}")
    print(f"  Std magnitude: {np.std(latents):.4f}")
    print(f"  Dynamic range: {max_magnitude/np.mean(np.abs(latents)):.2f}x")
    
    return latents

In [ ]:
# Setup model
print("Loading EnCodec model...")
model = EncodecModel.from_pretrained("facebook/encodec_24khz")
model.config.target_bandwidths = [6.0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

print(f"Model training mode: {model.training}")

In [ ]:

    
if file_name == None:
    file_path = pick_file()
    if file_path:
        full_path = Path(file_path)
        print(f"Selected: {full_path}")
    else:
        print("No file selected")

load_and_examine_tokens(full_path)

In [ ]:
audio_codes, audio_scales, padding_mask = load_and_examine_tokens(full_path)

# Move data to same device as model  
audio_codes = audio_codes.to(device)
if isinstance(audio_scales, list):
    audio_scales = [scale.to(device) if hasattr(scale, 'to') else scale for scale in audio_scales]
elif hasattr(audio_scales, 'to'):
    audio_scales = audio_scales.to(device)
padding_mask = padding_mask.to(device)

In [ ]:
# Reconstruct with different codebook counts
print(f"\n🔊 Reconstructing audio...")
reconstructions = reconstruct_audio_progressive(audio_codes, audio_scales, padding_mask, model, [2, 4, 8])#[2, 4, 8])

# Plot comparisons
print(f"\n📈 Plotting waveform comparisons...")
plot_audio_comparisons(reconstructions)

In [ ]:
    print(f"\n📊 Plotting spectrograms...")
    plot_spectrograms(reconstructions)
    
    # Audio controls
    create_audio_controls(reconstructions)

In [ ]:
from rnencodec.generator import RNNGenerator
from rnencodec.model.gru_audio_model import RNN, GRUModelConfig
from rnencodec.audioDataLoader.audio_dataset import LatentDatasetConfig
from rnencodec.audioDataLoader.audio_dataset import  preprocess_latents_for_RNN 


saved_configs = torch.load(config_path, weights_only=False)
model_config = GRUModelConfig(**saved_configs["model_config"])
data_config = LatentDatasetConfig(**saved_configs["data_config"])
clamp_val = data_config.clamp_val # need this to map between encodec latents and model input ranges


top_n = 24 #'Sample from the top N most likely outputs.'
temperature = .8 #'Controls the randomness of predictions.'
length_seconds =20.0 #'Length of the audio to generate in seconds.'
frame_rate=75
generation_length = int(length_seconds * frame_rate)
sr=24000
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device = 'cpu'
device

enc_model = EncodecModel.from_pretrained("facebook/encodec_24khz").to(device)
enc_model.eval()

#------------------------------------------
# need the model to check how many codebooks it was trained on and what the conditioning parameter length was
model = RNN(model_config, enc_model).to(device)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

n_q=model.config.n_q
cond_size=model.config.cond_size
print(f'nq={n_q} and cond_size={cond_size}')
#------------------------------------------
# Take first n codebooks and ensure batch dimension of original sound (eg chirp), get their latents, and run them through the water model. 
# -----------------------------------------
partial_codes = audio_codes[:, :n_q, :].to(device)  # Add batch dim 

latent_embeddings = efficient_codes_to_latents(enc_model, partial_codes).squeeze(0).t()
latent_embeddings = preprocess_latents_for_RNN(latent_embeddings, clamp_val)

print(f'latent_embeddings is {latent_embeddings.shape} (T, D)')
hops=latent_embeddings.shape[0] #Need the number of hops to construct a parameter vector that is the same length as the input sound. 
print(f'number of hops for constructing parameter vector = {hops}')

<div style="width: 100%; height: 30px; background-color: green;"></div>
<b>Build your own sequence</b> </br> 
The param sequence must fit the model!!

In [ ]:
#Used to spread a parameter vector over time - generates with unchanging parameter values.
def repeat_values(values, T, device=None, dtype=torch.float32):
    x = torch.tensor(values, dtype=dtype, device=device)
    return x.unsqueeze(0).expand(T, -1)

# creates the one-hot vector of classes
def one_hot(length, pos): # pos in [0, length-1]
    if not (0 <= pos < length):
        raise IndexError("pos out of range")
    return [1 if i == pos else 0 for i in range(length)]


<div style="width: 100%; height: 5px; background-color: green;"></div>
Construct parameter vector for model control

In [ ]:
# These are the parameters if you happen to have loaded a syntext7 trained model
# ['param 1', 'Chirps 1H', 'Applause 1H', 'Bugs 1H', 'Peepers 1H', 'Pistons 1H', 'Wind 1H', 'TokWotal 1H'  ]
# class number   0           1               2         3              4             5          6

In [ ]:
clslist=one_hot(7,6) #This model has one continuous parameter, and 7 class values.  
#clslist=[.0,1,.25,1,.25,.5,1]
paramlist=[.75, *clslist]
params_seq = repeat_values(paramlist, hops) # the second value is the number of hops we will run

print(f'(T, D) params_seq.shape is {params_seq.shape}')

In [ ]:
import time
bazrnn=RNNGenerator(model, model_config, data_config, enc_model, 2, 2) #, "sample", top_n, temperature)
       #RNNGenerator(inference_model, inference_model_config, testdata_config, enc_model, max_length, max_length)
#warm up prevents noise bursts at the begininng of the audio run
#filtered_audio = bazrnn.warmup(cond_seq, latent_seq=partial_codes)
start = time.perf_counter()
generated_audio=bazrnn.getNextAudioHop(params_seq, latent_seq=latent_embeddings ) 
elapsed = time.perf_counter() - start
print(f"Generated { generated_audio.shape[0]} samples  in {elapsed:.2f}s.")

In [ ]:
display(Audio(generated_audio, rate=sr))